#  Week 3, Day 1 — June 1, 2026
## Initialize GeoPandas & Build 1°×1° Coordinate Grid


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw": BASE_DIR+"/data/raw","processed": BASE_DIR+"/data/processed",
        "metadata": BASE_DIR+"/data/metadata","visualizations": BASE_DIR+"/visualizations"}
LAT_MIN,LAT_MAX,LON_MIN,LON_MAX = 5,30,65,95
YEAR_MIN,YEAR_MAX = 2015,2026
import geopandas as gpd
from shapely.geometry import box
print(f'GeoPandas version: {gpd.__version__} ✅')

## Step 1: Build the 1°×1° Grid

In [ ]:
# Create all grid cells in the bounding box
lat_steps = np.arange(LAT_MIN, LAT_MAX)   # 5,6,7,...,29 → 25 rows
lon_steps = np.arange(LON_MIN, LON_MAX)   # 65,66,...,94 → 30 cols

cells = []
for lat in lat_steps:
    for lon in lon_steps:
        cells.append({
            'grid_lat': lat,
            'grid_lon': lon,
            'geometry': box(lon, lat, lon+1, lat+1)  # (minx,miny,maxx,maxy)
        })

grid_gdf = gpd.GeoDataFrame(cells, crs='EPSG:4326')

print(f'Grid created: {len(grid_gdf)} cells')
print(f'  Lat range: {grid_gdf["grid_lat"].min()}–{grid_gdf["grid_lat"].max()} ({lat_steps.shape[0]} rows)')
print(f'  Lon range: {grid_gdf["grid_lon"].min()}–{grid_gdf["grid_lon"].max()} ({lon_steps.shape[0]} cols)')
print(f'  Each cell = 1°×1° ≈ 10,000–12,000 km² depending on latitude')
print()
print(grid_gdf.head(5))

## Step 2: Visualize the Grid

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
grid_gdf.boundary.plot(ax=ax, color='steelblue', linewidth=0.5, alpha=0.6)

# Highlight grid extent
from matplotlib.patches import FancyArrowPatch
ax.set_xlim(62, 98)
ax.set_ylim(2, 33)
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title(f'1°×1° Spatial Grid — Indian Ocean Bounding Box\n{len(grid_gdf)} cells covering Arabian Sea, Bay of Bengal, Andaman Sea', fontsize=13)
ax.grid(True, alpha=0.3)

# City markers
cities = {'Mumbai':(72.8,19.0),'Chennai':(80.3,13.0),'Kolkata':(88.4,22.6),'Kochi':(76.3,9.9)}
for city,(lon,lat) in cities.items():
    ax.plot(lon,lat,'r*',ms=10,zorder=5)
    ax.annotate(city,(lon,lat),textcoords='offset points',xytext=(5,5),fontsize=8)

plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week3_Day1_spatial_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grid visualization saved ✅')

In [ ]:
# Save grid for use in subsequent days
grid_gdf.to_file(DIRS['processed']+'/spatial_grid.geojson', driver='GeoJSON')
print(f'Grid saved: spatial_grid.geojson ({len(grid_gdf)} cells) ✅')